# MIND Recommender — Model Loader

Defines a scorer abstraction that the diversity re-ranking pipeline depends on.
Any model — single run or ensemble — that satisfies the `BaseScorer` contract
can be dropped into `reranking_diversity_refactored.ipynb` unchanged.

```
BaseScorer  (ABC)
├── NeuralScorer    wraps any single NRMS / NAML / LSTUR / NPA checkpoint
│     NeuralScorer.from_run(run_id, ctx)  ← reads sweep_summary.csv
└── EnsembleScorer  weighted combination of any BaseScorer instances
      EnsembleScorer([scorer_a, scorer_b], weights=[0.6, 0.4])
```

**Pipeline contract** (the only two methods the re-ranker ever calls):
```python
profile = scorer.user_profile(history_ids, user_id=None)
score   = scorer.score(profile, article_id)  # → float
```

## 1. Imports & Device

In [ ]:
import sys, random
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

MIND_REC = Path(".").resolve()
if str(MIND_REC) not in sys.path:
    sys.path.insert(0, str(MIND_REC))

from config import Config
from data.dataset import parse_news, parse_behaviors
from data.vocab import Vocab
from models.nrms        import NRMS
from models.naml        import NAML
from models.lstur       import LSTUR
from models.npa         import NPA
from models.fastformer  import Fastformer

_MODEL_REGISTRY = {
    "nrms":        NRMS,
    "naml":        NAML,
    "lstur":       LSTUR,
    "npa":         NPA,
    "fastformer":  Fastformer,
}

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Device: {DEVICE}")

## 2. Data Loading

`DataContext` bundles everything derived from the raw MIND files.
It is shared across all scorers so data is only loaded once.

In [ ]:
@dataclass
class DataContext:
    """
    Shared data derived from MIND train + dev splits.
    Passed to NeuralScorer.from_run() so every scorer uses the same
    vocabulary, mappings, and article text — loaded exactly once.
    """
    all_news:   dict          # news_id → {title, abstract, category, subcategory}
    vocab:      Vocab
    cat2idx:    dict
    subcat2idx: dict
    user2idx:   dict          # raw user_id string → int index (LSTUR / NPA)
    cfg:        Config        # base config (data dims); models override model fields
    device:     torch.device
    sweep_df:   pd.DataFrame  # sweep_summary.csv, indexed for fast lookup


def build_data_context(
    train_dir: str | Path = "MINDsmall_train/MINDsmall_train",
    dev_dir:   str | Path = "MINDsmall_dev/MINDsmall_dev",
    sweep_csv: str | Path = "results/sweep/sweep_summary.csv",
    device:    torch.device = DEVICE,
) -> DataContext:
    """Load MIND data once and return a DataContext shared by all scorers."""
    print("Loading news...")
    train_news = parse_news(str(Path(train_dir) / "news.tsv"))
    dev_news   = parse_news(str(Path(dev_dir)   / "news.tsv"))
    all_news   = {**train_news, **dev_news}
    print(f"  train={len(train_news):,}  dev={len(dev_news):,}  total={len(all_news):,}")

    print("Loading behaviors...")
    train_beh = parse_behaviors(str(Path(train_dir) / "behaviors.tsv"))
    dev_beh   = parse_behaviors(str(Path(dev_dir)   / "behaviors.tsv"))
    all_beh   = train_beh + dev_beh

    cats       = sorted({v["category"]    for v in all_news.values()})
    subcats    = sorted({v["subcategory"] for v in all_news.values()})
    cat2idx    = {c: i + 1 for i, c in enumerate(cats)}
    subcat2idx = {s: i + 1 for i, s in enumerate(subcats)}

    users    = sorted({b["user_id"] for b in all_beh})
    user2idx = {u: i + 1 for i, u in enumerate(users)}

    print("Building vocabulary...")
    vocab = Vocab()
    vocab.build(
        [n["title"] + " " + n["abstract"] for n in all_news.values()],
        min_freq=1,
    )

    cfg = Config()
    cfg.model.num_users = len(user2idx)

    sweep_df = pd.read_csv(sweep_csv)

    print(f"  vocab={len(vocab):,}  users={len(user2idx):,}  "
          f"cats={len(cat2idx)}  subcats={len(subcat2idx)}")
    return DataContext(
        all_news=all_news, vocab=vocab, cat2idx=cat2idx,
        subcat2idx=subcat2idx, user2idx=user2idx,
        cfg=cfg, device=device, sweep_df=sweep_df,
    )


ctx = build_data_context()

## 3. BaseScorer — the reranking contract

Every scorer — single model, ensemble, or any future variant — must implement
these two methods. The re-ranking functions (`rerank_entity_mmr`, `run_sweep`,
etc.) only ever call these two methods, so any `BaseScorer` subclass works
as a drop-in.

In [ ]:
class BaseScorer(ABC):
    """
    Contract every scorer must satisfy for the diversity re-ranking pipeline.

    The `profile` object returned by user_profile() is opaque to the pipeline —
    it can be a numpy vector, a list of vectors (ensemble), or anything else,
    as long as score() knows how to consume it.
    """

    @abstractmethod
    def user_profile(self, history_ids, user_id: Optional[str] = None):
        """
        Build a user representation from click history.

        Parameters
        ----------
        history_ids : iterable[str]
            Ordered news IDs the user has clicked (most recent last).
        user_id : str, optional
            Raw user ID (e.g. "U13740").  Used by LSTUR / NPA for
            personalised attention; ignored by NRMS / NAML.

        Returns
        -------
        profile
            An opaque object passed unchanged to score().
        """

    @abstractmethod
    def score(self, profile, article_id: str) -> float:
        """
        Relevance of `article_id` for the user represented by `profile`.

        Returns
        -------
        float
            Higher = more relevant.  Scale is scorer-specific (the re-ranker
            normalises internally before computing diversity bonuses).
        """

print("BaseScorer defined.")

## 4. NeuralScorer

Wraps any single NRMS / NAML / LSTUR / NPA checkpoint.

- `user_profile()` → 1-D numpy vector (user embedding)
- `score()` → dot product with precomputed article embedding

Use the `from_run(run_id, ctx)` factory to load by run ID — it reads
`sweep_summary.csv`, instantiates the right model class, loads weights,
and precomputes all article embeddings.

In [ ]:
class NeuralScorer(BaseScorer):
    """
    BaseScorer backed by a single trained neural model.

    Do not construct directly — use NeuralScorer.from_run(run_id, ctx).
    """

    def __init__(
        self,
        model:          torch.nn.Module,
        article_embeds: dict,          # news_id → (news_dim,) float32
        ctx:            DataContext,
        news_dim:       int,
        needs_user_idx: bool,          # True for LSTUR / NPA
        run_meta:       dict,          # for display / inspection
    ):
        self._model          = model
        self._article_embeds = article_embeds
        self._ctx            = ctx
        self._news_dim       = news_dim
        self._needs_user_idx = needs_user_idx
        self.meta            = run_meta

    # ── public contract ────────────────────────────────────────────────────

    def user_profile(self, history_ids, user_id: Optional[str] = None) -> np.ndarray:
        history = list(history_ids)
        if not history:
            return np.zeros(self._news_dim, dtype=np.float32)
        return self._encode_user(history, user_id)

    def score(self, profile: np.ndarray, article_id: str) -> float:
        news_vec = self._article_embeds.get(article_id)
        if news_vec is None:
            return 0.0
        return float(np.dot(profile, news_vec))

    # ── inspection helpers ─────────────────────────────────────────────────

    @property
    def article_embeds(self) -> dict:
        """news_id → embedding — exposed for vectorised ops if needed."""
        return self._article_embeds

    @property
    def news_dim(self) -> int:
        return self._news_dim

    def __repr__(self):
        m = self.meta
        return (
            f"NeuralScorer(run={m.get('run_id')}, model={m.get('model')}, "
            f"news_dim={self._news_dim}, "
            f"val_auc={m.get('best_val_auc', float('nan')):.4f})"
        )

    # ── factory ────────────────────────────────────────────────────────────

    @classmethod
    def from_run(cls, run_id: int, ctx: DataContext, batch_size: int = 512) -> "NeuralScorer":
        """
        Load a checkpoint by run_id from sweep_summary.csv.

        Reads hyperparameters, instantiates the correct model class (NRMS /
        NAML / LSTUR / NPA / Fastformer), loads weights, and precomputes
        article embeddings.
        """
        row = ctx.sweep_df[ctx.sweep_df["run_id"] == run_id]
        assert len(row) == 1, f"run_id={run_id} not found in sweep_summary.csv"
        run = row.iloc[0]

        model_name = run["model"]
        ckpt_path  = Path(str(run["checkpoint"]).replace("\\", "/"))
        if not ckpt_path.is_absolute():
            ckpt_path = MIND_REC / ckpt_path
        assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"

        # ── build config from sweep row ───────────────────────────────────
        cfg = Config()
        cfg.model.dropout    = float(run["dropout"])
        cfg.model.num_users  = len(ctx.user2idx)
        cfg.train.model_name = model_name
        cfg.train.lr         = float(run["lr"])

        def _int(val):
            try:
                v = float(val)
                return int(v) if not np.isnan(v) else None
            except (TypeError, ValueError):
                return None

        if _int(run.get("num_filters")):
            cfg.model.num_filters = _int(run["num_filters"])
        if _int(run.get("cnn_kernel_size")):
            cfg.model.cnn_kernel_size = _int(run["cnn_kernel_size"])
        if str(run.get("lstur_mode", "nan")) != "nan":
            cfg.model.lstur_mode = str(run["lstur_mode"])
        if _int(run.get("batch_size")):
            cfg.train.batch_size = _int(run["batch_size"])

        needs_user_idx = model_name in ("lstur", "npa")

        # nrms and fastformer both use num_heads * head_dim as news_dim
        if model_name in ("nrms", "fastformer"):
            cfg.model.num_heads = _int(run["num_heads"]) or cfg.model.num_heads
            cfg.model.head_dim  = _int(run["head_dim"])  or cfg.model.head_dim
            news_dim = cfg.model.num_heads * cfg.model.head_dim
        else:
            news_dim = cfg.model.num_filters

        # ── instantiate model ─────────────────────────────────────────────
        n_vocab   = len(ctx.vocab)
        n_cats    = len(ctx.cat2idx)
        n_subcats = len(ctx.subcat2idx)
        n_users   = len(ctx.user2idx)

        if model_name == "nrms":
            nn_model = NRMS(n_vocab, cfg)
        elif model_name == "naml":
            nn_model = NAML(n_vocab, n_cats, n_subcats, cfg)
        elif model_name == "lstur":
            nn_model = LSTUR(n_vocab, n_users, cfg)
        elif model_name == "npa":
            nn_model = NPA(n_vocab, n_users, cfg)
        elif model_name == "fastformer":
            nn_model = Fastformer(n_vocab, cfg)
        else:
            raise ValueError(f"Unknown model: {model_name}")

        state = torch.load(ckpt_path, map_location=ctx.device)
        nn_model.load_state_dict(state)
        nn_model = nn_model.to(ctx.device)
        nn_model.eval()

        n_params = sum(p.numel() for p in nn_model.parameters() if p.requires_grad)
        print(f"Loaded run {run_id}: {model_name.upper()}  "
              f"news_dim={news_dim}  params={n_params:,}  "
              f"val_auc={run['best_val_auc']:.4f}  test_auc={run['test_auc']:.4f}")

        # ── precompute article embeddings ─────────────────────────────────
        all_ids  = list(ctx.all_news.keys())
        emb_list = []
        for start in tqdm(range(0, len(all_ids), batch_size), desc=f"  Encoding ({model_name})"):
            batch = all_ids[start : start + batch_size]
            emb_list.append(
                _encode_news_batch(batch, nn_model, cfg, ctx, needs_user_idx)
            )
        emb_matrix = np.concatenate(emb_list, axis=0)   # (N, news_dim)
        article_embeds = {nid: emb_matrix[i] for i, nid in enumerate(all_ids)}

        meta = {
            "run_id":        int(run_id),
            "model":         model_name,
            "best_val_auc":  float(run["best_val_auc"]),
            "test_auc":      float(run["test_auc"]),
            "checkpoint":    str(ckpt_path),
        }
        return cls(nn_model, article_embeds, ctx, news_dim, needs_user_idx, meta)

    # ── private encoding ───────────────────────────────────────────────────

    @torch.no_grad()
    def _encode_user(self, history_ids: list, user_id: Optional[str]) -> np.ndarray:
        ctx = self._ctx
        H   = ctx.cfg.data.max_history
        hist      = history_ids[-H:]
        pad_len   = H - len(hist)
        hist_pad  = hist + [""] * pad_len
        mask      = [True] * len(hist) + [False] * pad_len

        encoded = [_encode_one_news(nid, ctx) for nid in hist_pad]
        dev = ctx.device

        titles    = torch.tensor([[e["title"]       for e in encoded]], dtype=torch.long,  device=dev)
        abstracts = torch.tensor([[e["abstract"]    for e in encoded]], dtype=torch.long,  device=dev)
        cats_t    = torch.tensor([[e["category"]    for e in encoded]], dtype=torch.long,  device=dev)
        subcats_t = torch.tensor([[e["subcategory"] for e in encoded]], dtype=torch.long,  device=dev)
        mask_t    = torch.tensor([mask],                               dtype=torch.bool,   device=dev)

        B, Hlen = 1, H
        self._model.eval()
        hist_vecs = self._model.encode_news(
            titles.view(B * Hlen, -1),
            abstracts.view(B * Hlen, -1),
            cats_t.view(B * Hlen),
            subcats_t.view(B * Hlen),
        ).view(B, Hlen, -1)

        uid_tensor = None
        if self._needs_user_idx:
            uid = ctx.user2idx.get(user_id, 0)
            uid_tensor = torch.tensor([uid], dtype=torch.long, device=dev)

        user_vec = self._model.encode_user(hist_vecs, mask_t, user_idx=uid_tensor)
        return user_vec.squeeze(0).cpu().numpy().astype(np.float32)


# ── module-level helpers used by the factory ───────────────────────────────

def _encode_one_news(nid: str, ctx: DataContext) -> dict:
    n = ctx.all_news.get(nid, {"category": "", "subcategory": "", "title": "", "abstract": ""})
    return {
        "title":       ctx.vocab.encode(n["title"],    ctx.cfg.data.max_title_len),
        "abstract":    ctx.vocab.encode(n["abstract"],  ctx.cfg.data.max_abstract_len),
        "category":    ctx.cat2idx.get(n["category"], 0),
        "subcategory": ctx.subcat2idx.get(n["subcategory"], 0),
    }


@torch.no_grad()
def _encode_news_batch(
    news_ids: list,
    model:    torch.nn.Module,
    cfg:      Config,
    ctx:      DataContext,
    needs_user_idx: bool,
) -> np.ndarray:
    """
    Encode a batch of articles → (B, news_dim) float32.
    user_idx=None: NPA falls back to zero attention (user-agnostic);
    LSTUR ignores user_idx for news encoding entirely.
    """
    model.eval()
    encoded   = [_encode_one_news(nid, ctx) for nid in news_ids]
    dev = ctx.device

    titles    = torch.tensor([e["title"]       for e in encoded], dtype=torch.long,  device=dev)
    abstracts = torch.tensor([e["abstract"]    for e in encoded], dtype=torch.long,  device=dev)
    cats_t    = torch.tensor([e["category"]    for e in encoded], dtype=torch.long,  device=dev)
    subcats_t = torch.tensor([e["subcategory"] for e in encoded], dtype=torch.long,  device=dev)

    vecs = model.encode_news(titles, abstracts, cats_t, subcats_t)
    return vecs.cpu().numpy().astype(np.float32)


print("NeuralScorer defined.")

## 5. EnsembleScorer

Combines any number of `BaseScorer` instances — neural models, heuristics, or
other ensembles — with optional per-scorer weights.

`user_profile()` returns a list of sub-profiles (one per sub-scorer).  
`score()` computes each sub-score and returns a weighted average.

The re-ranker sees only `scorer.score(profile, nid)` — it never inspects the
internal structure, so the ensemble is fully transparent to the pipeline.

In [ ]:
class EnsembleScorer(BaseScorer):
    """
    Weighted combination of any BaseScorer instances.

    Parameters
    ----------
    scorers : list[BaseScorer]
        Sub-scorers to combine.  Any BaseScorer subclass works.
    weights : list[float], optional
        Per-scorer weights.  Defaults to uniform.  Need not sum to 1
        (normalised internally).
    """

    def __init__(self, scorers: List[BaseScorer], weights: Optional[List[float]] = None):
        assert len(scorers) >= 2, "EnsembleScorer needs at least two sub-scorers"
        self._scorers = scorers
        if weights is None:
            weights = [1.0] * len(scorers)
        total = sum(weights)
        self._weights = [w / total for w in weights]   # normalise

    def user_profile(self, history_ids, user_id: Optional[str] = None) -> list:
        """
        Returns a list of sub-profiles, one per sub-scorer.
        The list is passed unchanged to score().
        """
        return [s.user_profile(history_ids, user_id=user_id) for s in self._scorers]

    def score(self, profile: list, article_id: str) -> float:
        """
        Weighted average of sub-scorer dot products.
        """
        return float(sum(
            w * s.score(p, article_id)
            for s, p, w in zip(self._scorers, profile, self._weights)
        ))

    def __repr__(self):
        parts = ", ".join(
            f"{repr(s)} × {w:.2f}"
            for s, w in zip(self._scorers, self._weights)
        )
        return f"EnsembleScorer([{parts}])"


print("EnsembleScorer defined.")

## 6. Instantiate Your Scorer

Edit this cell to choose what feeds into the re-ranker.

**Option A — single best run:**
```python
scorer = NeuralScorer.from_run(20, ctx)
```

**Option B — ensemble of two runs:**
```python
s13 = NeuralScorer.from_run(13, ctx)
s20 = NeuralScorer.from_run(20, ctx)
scorer = EnsembleScorer([s13, s20], weights=[0.4, 0.6])
```

**Option C — ensemble across architectures:**
```python
nrms  = NeuralScorer.from_run(20,  ctx)   # best NRMS
lstur = NeuralScorer.from_run(5,   ctx)   # best LSTUR
npa   = NeuralScorer.from_run(10,  ctx)   # best NPA
scorer = EnsembleScorer([nrms, lstur, npa], weights=[0.5, 0.25, 0.25])
```

In [ ]:
# ── Choose your scorer here ──────────────────────────────────────────────────

scorer = NeuralScorer.from_run(20, ctx)   # single best NRMS run

# ─── Uncomment for an ensemble ───────────────────────────────────────────────
# s13 = NeuralScorer.from_run(13, ctx)
# s20 = NeuralScorer.from_run(20, ctx)
# scorer = EnsembleScorer([s13, s20], weights=[0.4, 0.6])

# ────────────────────────────────────────────────────────────────────────────
print()
print(repr(scorer))

## 7. Sanity Checks

In [ ]:
from data.dataset import parse_behaviors as _pb
_train_beh = _pb("MINDsmall_train/MINDsmall_train/behaviors.tsv")
_dev_beh   = _pb("MINDsmall_dev/MINDsmall_dev/behaviors.tsv")

sample_beh = next(b for b in _train_beh if b["history"] and b["pos"])
sample_uid = sample_beh["user_id"]
sample_hist = sample_beh["history"][:10]
all_news_ids = list(ctx.all_news.keys())

# Check 1: user_profile returns finite values
profile = scorer.user_profile(sample_hist, user_id=sample_uid)
# For ensemble profile is a list; for single scorer it's a numpy array
profiles = profile if isinstance(profile, list) else [profile]
for p in profiles:
    assert np.isfinite(p).all(), "NaN/Inf in user profile"
print("[✓] user_profile: finite values")

# Check 2: score returns a finite float
s = scorer.score(profile, all_news_ids[0])
assert isinstance(s, float) and np.isfinite(s), "score() must return finite float"
print(f"[✓] scorer.score({all_news_ids[0]}) = {s:.4f}")

# Check 3: unknown article → 0.0 (no crash)
s_unknown = scorer.score(profile, "N_DOES_NOT_EXIST")
assert s_unknown == 0.0, "Unknown article should score 0.0"
print("[✓] Unknown article → 0.0")

# Check 4: clicked articles score higher than random on average
clicked_ids = sample_beh["pos"][:5]
random_ids  = random.sample(
    [n for n in all_news_ids if n not in set(clicked_ids)], 20
)
clicked_scores = [scorer.score(profile, n) for n in clicked_ids]
random_scores  = [scorer.score(profile, n) for n in random_ids]
print(f"[·] Mean clicked score : {np.mean(clicked_scores):.4f}")
print(f"[·] Mean random  score : {np.mean(random_scores):.4f}")

## 8. Integration

In `reranking_diversity_refactored.ipynb`, cell 19 now contains:
```python
%run nrms_model_loader.ipynb
```
which runs this notebook and makes `scorer`, `ctx`, and the helper classes
available in that namespace.

All re-ranking functions (`rerank_entity_mmr`, `rerank_embedding_mmr`,
`rerank_hybrid_mmr`, `rerank_calibrated`, `run_sweep`, `eval_event`) call only:
```python
profile = scorer.user_profile(prior, user_id=uid)
score   = scorer.score(profile, candidate_id)
```
Nothing else in the re-ranking notebook needs to change.

### Variable name guide

| Name | From | Used for |
|---|---|---|
| `scorer` | this notebook | relevance scoring in MMR |
| `embed_map` | reranking notebook cell 22 | MiniLM semantic embeddings for embedding diversity |
| `ctx` | this notebook | shared data (vocab, mappings, news) |
| `DataContext` | this notebook | type for the shared data bundle |
| `NeuralScorer` | this notebook | load any single checkpoint |
| `EnsembleScorer` | this notebook | combine multiple scorers |

In [ ]:
# Demo: rank a full impression from the dev set
demo_beh    = next(b for b in _dev_beh if b["history"] and b["pos"])
demo_uid    = demo_beh["user_id"]
demo_cands  = [imp.split("-")[0] for imp in demo_beh["impressions"]]
demo_labels = [int(imp.split("-")[1]) for imp in demo_beh["impressions"]]

demo_profile = scorer.user_profile(demo_beh["history"], user_id=demo_uid)
demo_scores  = {nid: scorer.score(demo_profile, nid) for nid in demo_cands}
demo_ranked  = sorted(demo_scores, key=lambda n: -demo_scores[n])

print(f"Scorer: {repr(scorer)}")
print("Top-5 ranked  [1=clicked, 0=not clicked]:")
for rank, nid in enumerate(demo_ranked[:5], 1):
    label = demo_labels[demo_cands.index(nid)]
    title = ctx.all_news.get(nid, {}).get("title", "<unknown>")[:65]
    print(f"  {rank}. [{label}] {title}  ({demo_scores[nid]:.4f})")